In [1]:
import torch

x = torch.ones(5)  # input tensor
y = torch.zeros(3)  # expected output
w = torch.randn(5, 3, requires_grad=True)
b = torch.randn(3, requires_grad=True)
z = torch.matmul(x, w)+b
loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y)

In [ ]:
# 변화도(Gradient) 계산하기
loss.backward()
print(w.grad)
print(b.grad)

tensor([[2.5744e-01, 2.1507e-04, 2.0278e-01],
        [2.5744e-01, 2.1507e-04, 2.0278e-01],
        [2.5744e-01, 2.1507e-04, 2.0278e-01],
        [2.5744e-01, 2.1507e-04, 2.0278e-01],
        [2.5744e-01, 2.1507e-04, 2.0278e-01]])
tensor([2.5744e-01, 2.1507e-04, 2.0278e-01])


In [ ]:
# 변화도 추적 멈추기
z = torch.matmul(x, w)+b
print(z.requires_grad)

with torch.no_grad():
    z = torch.matmul(x, w)+b
print(z.requires_grad)

z = torch.matmul(x, w)+b
z_det = z.detach()
print(z_det.requires_grad)



True
False


변화도 추적을 멈추는 이유
* 신경망의 일부 매개변수를 고정된 매개변수(frozen parameter)로 표시합니다.
* 변화도를 추적하지 않는 텐서의 연산이 더 효율적이기 때문에, 순전파 단계만 수행할 때 연산 속도가 향상됩니다.

### 연산 그래프에 대한 추가 정보
* autograd는 데이터(텐서)의 및 실행된 모든 연산들(및 연산 결과가 새로운 텐서인 경우도 포함하여)의 기록을 Function 객체로 구성된 방향성 비순환 그래프(DAG; Directed Acyclic Graph)에 저장(keep)합니다.
* 이 방향성 비순환 그래프(DAG)의 잎(leave)은 입력 텐서이고, 뿌리(root)는 결과 텐서입니다. 
* 연쇄 법칙(chain rule)에 따라 변화도를 자동으로 계산할 수 있습니다.

* 순전파에서
    * 요청된 연산을 수행하여 결과 텐서를 계산하고,
    * DAG에 연산의 변화도 기능(gradient function) 를 유지(maintain)합니다.
* 역전파에서
    * 각 .grad_fn 으로부터 변화도를 계산하고,
    * 각 텐서의 .grad 속성에 계산 결과를 쌓고(accumulate),
    * 연쇄 법칙을 사용하여, 모든 잎(leaf) 텐서들까지 전파(propagate)합니다.

* 매번 .backward() 가 호출되고 나면, autograd는 새로운 그래프를 채우기(populate) 시작합니다. 이러한 점 덕분에 모델에서 흐름 제어(control flow) 구문들을 사용할 수 있게 되는 것이다.

### 선택적으로 읽기(Optional Reading): 텐서 변화도와 야코비안 곱 (Jacobian Product)
backward() 출력 함수가 임의의 텐서이면 실제 변화도가 아닌 야코비안 곱을 계산한다. 원래 크기는 곱을 계산하려고 하는 원래 텐서의 크기와 같아야 한다.

In [5]:
inp = torch.eye(4, 5, requires_grad=True)
out = (inp+1).pow(2).t()
out.backward(torch.ones_like(out), retain_graph=True)
print(f"First call\n{inp.grad}")
out.backward(torch.ones_like(out), retain_graph=True)
print(f"\nSecond call\n{inp.grad}")
inp.grad.zero_()
out.backward(torch.ones_like(out), retain_graph=True)
print(f"\nCall after zeroing gradients\n{inp.grad}")

First call
tensor([[4., 2., 2., 2., 2.],
        [2., 4., 2., 2., 2.],
        [2., 2., 4., 2., 2.],
        [2., 2., 2., 4., 2.]])

Second call
tensor([[8., 4., 4., 4., 4.],
        [4., 8., 4., 4., 4.],
        [4., 4., 8., 4., 4.],
        [4., 4., 4., 8., 4.]])

Call after zeroing gradients
tensor([[4., 2., 2., 2., 2.],
        [2., 4., 2., 2., 2.],
        [2., 2., 4., 2., 2.],
        [2., 2., 2., 4., 2.]])
